# CUAD ONNX Export — fp32 → INT8 quantization + evaluation (Colab GPU)

Downloads the `cuad-extractor` bundle from the MLflow registry (produced by
`cuad_finetune.ipynb`), exports it to ONNX fp32 via Optimum, quantizes to
dynamic-INT8, evaluates with CUAD metrics (F1 / ANLS / AUPR) and OCR CER,
logs results to MLflow, then registers the INT8 bundle as `cuad-extractor-onnx-int8`.

**Runtime:** `Runtime > Change runtime type > GPU (T4 or better)`.

## 1. Install

In [ ]:
# Use a GPU runtime: Runtime > Change runtime type > GPU.
!pip install -q "docintel[train,kie] @ git+https://github.com/KhoiDang1209/AI-Document-Understanding.git@master#subdirectory=docintel"
!pip install -q optimum[onnxruntime]

## 2. Setup

In [ ]:
from pathlib import Path

import mlflow

from docintel.optimize.export import download_registered_model, export_qa_to_onnx
from docintel.optimize.quantize import quantize_dynamic_int8

TRACKING_URI = "file:./mlruns"  # local file-store; change to remote URI if needed
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("cuad-onnx-export")

REGISTERED_MODEL = "cuad-extractor"
MODEL_VERSION = "1"  # update if you have multiple training runs

## 3. Download the registered PyTorch bundle

In [ ]:
bundle_dir = Path("cuad-extractor-bundle")
bundle = download_registered_model(
    name=REGISTERED_MODEL,
    version=MODEL_VERSION,
    dest=bundle_dir,
    tracking_uri=TRACKING_URI,
)
print("Downloaded bundle:", bundle)

## 4. Export to ONNX fp32

In [ ]:
fp32_dir = Path("cuad-extractor-onnx-fp32")
fp32_dir = export_qa_to_onnx(model_dir=bundle / "model", out_dir=fp32_dir)
print("fp32 ONNX saved to:", fp32_dir)

## 5. Quantize to INT8

In [ ]:
int8_dir = Path("cuad-extractor-onnx-int8")
int8_dir = quantize_dynamic_int8(onnx_dir=fp32_dir, out_dir=int8_dir)
print("INT8 ONNX saved to:", int8_dir)

## 6. Evaluate: F1 / ANLS / AUPR on clean text + OCR CER

In [ ]:
import numpy as np
import onnxruntime as ort  # type: ignore
from datasets import load_dataset
from transformers import AutoTokenizer  # type: ignore

from docintel.contracts.aggregate import WindowSpan, aggregate_clause, best_spans_from_window
from docintel.contracts.eval import anls, average_precision, token_f1
from docintel.contracts.ocr_cer import cer

DATASET_REVISION = "main"
raw = load_dataset("theatticusproject/cuad-qa", revision=DATASET_REVISION, split="test")

onnx_path = next(int8_dir.rglob("*quantized*.onnx"), None) or next(int8_dir.rglob("*.onnx"))
session = ort.InferenceSession(str(onnx_path))
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

# Capture the input names the graph actually declares so we feed only those.
_input_names: frozenset[str] = frozenset(inp.name for inp in session.get_inputs())
_CANDIDATE_INPUTS = ("input_ids", "attention_mask", "token_type_ids")

MAX_SEQ_LENGTH = 512
DOC_STRIDE = 128
N_BEST = 10
MAX_ANSWER_LENGTH = 200
NO_ANSWER_THRESHOLD = 0.0


def _predict(question: str, context: str) -> str:
    """Sliding-window prediction matching the serving path in CuadQaOnnxExtractor."""
    enc = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="np",
    )
    spans: list[WindowSpan] = []
    num_windows = enc["input_ids"].shape[0]
    for i in range(num_windows):
        feeds = {
            name: enc[name][i : i + 1].astype(np.int64)
            for name in _CANDIDATE_INPUTS
            if name in _input_names and name in enc
        }
        start_logits, end_logits = session.run(None, feeds)
        offsets = [tuple(o) for o in enc["offset_mapping"][i]]
        spans.extend(
            best_spans_from_window(
                start_logits[0],
                end_logits[0],
                offsets,
                N_BEST,
                MAX_ANSWER_LENGTH,
            )
        )
    # aggregate_clause returns ExtractedClause list; take best answer text
    results = aggregate_clause("_eval", context, spans, N_BEST, NO_ANSWER_THRESHOLD)
    return results[0].answer_text if results else ""


# Evaluate on a sample (full eval is slow; increase N for thoroughness)
N = 200
sample = raw.select(range(min(N, len(raw))))

f1_scores, anls_scores, labels, confs = [], [], [], []
for ex in sample:
    gold_answers = ex["answers"]["text"]
    if not gold_answers:
        gold = ""
        labels.append(0)
    else:
        gold = gold_answers[0]
        labels.append(1)
    pred = _predict(ex["question"], ex["context"])
    f1_scores.append(token_f1(pred, gold))
    anls_scores.append(anls(pred, gold))
    confs.append(1.0 if pred else 0.0)

mean_f1 = float(np.mean(f1_scores))
mean_anls = float(np.mean(anls_scores))
aupr = average_precision(confs, labels) if any(labels) else 0.0

# OCR CER: compare digital text with a dummy OCR hypothesis (illustrative).
# In practice, replace `ocr_hypothesis` with actual OCR output from the ingest path.
sample_context = sample[0]["context"]
ocr_hypothesis = sample_context  # identity -> CER=0 as baseline
mean_cer = cer(sample_context, ocr_hypothesis)

print(f"mean F1:   {mean_f1:.4f}")
print(f"mean ANLS: {mean_anls:.4f}")
print(f"AUPR:      {aupr:.4f}")
print(f"OCR CER:   {mean_cer:.4f}")

## 7. Log metrics to MLflow

In [ ]:
with mlflow.start_run(run_name="cuad-int8-eval"):
    mlflow.log_params({"base_model": REGISTERED_MODEL, "version": MODEL_VERSION, "eval_n": N})
    mlflow.log_metrics(
        {"mean_f1": mean_f1, "mean_anls": mean_anls, "aupr": aupr, "ocr_cer": mean_cer}
    )
    mlflow.log_artifacts(str(int8_dir), artifact_path="int8_bundle")
    print("Metrics and INT8 bundle logged to MLflow")

## 8. Register INT8 bundle as `cuad-extractor-onnx-int8`

In [ ]:
runs = mlflow.search_runs(experiment_names=["cuad-onnx-export"], order_by=["start_time DESC"])
run_id = runs.iloc[0]["run_id"]
artifact_uri = f"runs:/{run_id}/int8_bundle"
mv = mlflow.register_model(artifact_uri, "cuad-extractor-onnx-int8")
print(f"Registered cuad-extractor-onnx-int8 version {mv.version}")

## 9. Serve locally on the laptop

Download the `cuad-extractor-onnx-int8` bundle, unzip it, then set:
```
DOCINTEL_CONTRACT_ONNX_LOCAL_PATH=/path/to/cuad-extractor-onnx-int8
```
Start the API (`uvicorn docintel.api.main:app --reload`) and post a PDF:
```bash
curl -X POST http://localhost:8000/contracts/extract \
     -F 'file=@contract.pdf;type=application/pdf'
```
Retrieve a stored contract:
```bash
curl http://localhost:8000/contracts/{id}
```